## 1. Imports


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")

## 2. Load Data


In [2]:
df = pd.read_csv(r"C:\Users\Naz\Desktop\Naz\jobs\inv\Code\fabenode-data-report\data\raw\user_48_feb02_14_full_hrv.csv")


## 3. Datetime Preparation


In [3]:
df["start_time"]=pd.to_datetime(df["start_time"])
df["finish_time"]=pd.to_datetime(df["finish_time"])


## 4. Pipeline Columns Overview


In [4]:
quality_cols = [
    "is_ekg_move_quality",
    "ekg_move_ratio",
    "is_ekg_quality",
    "ekg_signal_quality_ratio",
    "ekg_quality_threshold",
    "ekg_peak_quality_ratio",
    "ekg_signal_length",
    "state",
    "message"]

## 5. Signal Quality Summary


In [5]:
df[quality_cols].head().T

,0,1,2,3,4
is_ekg_move_quality,True,True,True,True,True
ekg_move_ratio,0.969,0.983,0.983,0.994,0.997
is_ekg_quality,False,False,False,False,False
ekg_signal_quality_ratio,0.0,0.0,0.0,0.0,0.0
ekg_quality_threshold,0.2,0.2,0.2,0.2,0.2
ekg_peak_quality_ratio,0.03871,0.031646,0.033684,0.020921,0.016878
ekg_signal_length,184352.0,184352.0,184352.0,184352.0,184320.0
state,SUCCESS,SUCCESS,SUCCESS,SUCCESS,SUCCESS
message,ERROR - ekg_quality_signal_filter_0.0,ERROR - ekg_quality_signal_filter_0.0,ERROR - ekg_quality_signal_filter_0.0,ERROR - ekg_quality_signal_filter_0.0,ERROR - ekg_quality_signal_filter_0.0


In [6]:
df["is_ekg_quality"].value_counts(dropna=False)

is_ekg_quality
True     1905
False     877
NaN       355
Name: count, dtype: int64

In [7]:
df["is_ekg_move_quality"].value_counts(dropna=False)

is_ekg_move_quality
True    3038
NaN       99
Name: count, dtype: int64

In [8]:
df[["ekg_move_ratio", "ekg_signal_quality_ratio", "ekg_peak_quality_ratio"]].describe()

,ekg_move_ratio,ekg_signal_quality_ratio,ekg_peak_quality_ratio
count,3038.000000,2782.000000,2782.000000
mean,0.965105,0.574136,0.495043
std,0.061802,0.411573,0.318431
min,0.311000,0.000000,0.006814
25%,0.961750,0.000000,0.212894
50%,0.989000,0.755500,0.484817
75%,1.000000,0.947000,0.808182
max,1.000000,0.997000,1.000000


#### Quality Columns Quick Analysis

- `ekg_signal_quality_ratio`
  - 25% = 0 → at least 25% of values are 0 (many poor signals)
  - Median (50%) = 0.755 → distribution is skewed (many low + many high values)
  - Mean < median → zeros pull average down

- `ekg_peak_quality_ratio`
  - Mean ≈ median → more balanced distribution
  - Gradual spread (no large mass at 0)

- `ekg_move_ratio`
  - Very high overall (mean ≈ 0.97)
  - 25% already ≈ 0.96 → most values are high
  - Many values = 1 → near-perfect movement quality

- Counts differ → columns have different missing values

## 6. Message and State Inspection

In [9]:
df["state"].value_counts(dropna=False)

state
SUCCESS    3038
NaN          99
Name: count, dtype: int64

In [10]:
df[df["state"]=="SUCCESS"]["message"].nunique()

103

In [11]:
df[df["state"]=="SUCCESS"]["message"].unique()

array(['ERROR - ekg_quality_signal_filter_0.0',
       'ERROR - ekg_quality_signal_filter_0.05790935926376107',
       'ERROR - hr_process-ERROR : There is no best segment for hr calculation',
       'ERROR - ekg_quality_signal_filter_0.06640082465277777',
       'ERROR - Traceback (most recent call last):\n  File "D:\\Work\\invamar\\Back-end\\bsp\\analyze.py", line 316, in analyze_ekg\n    signals, peak_quality_ratio = calculate_real_signals(signals,filter_high_value=200)\n                                  ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File "D:\\Work\\invamar\\Back-end\\bsp\\quality.py", line 130, in calculate_real_signals\n    quality_with_nan, quality_with_zero = calculate_quality(ecg, filtered_peak_indices, sampling_rate=sampling_rate)\n                                          ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File "D:\\Work\\invamar\\Back-end\\bsp\\quality.py", line 29, in calculate_quality\n    heartbeats = n

In [12]:
df[df["state"]!="SUCCESS"]["message"].nunique()

7

In [13]:
df[df["state"]!="SUCCESS"]["message"].unique()

array([nan,
       'ERROR - movement_process-ERROR - ekg_quality_move_filter_0.144',
       'ERROR - movement_process-ERROR - ekg_quality_move_filter_0.003',
       'ERROR - movement_process-ERROR - ekg_quality_move_filter_0.039',
       'ERROR - movement_process-ERROR - ekg_quality_move_filter_0.206',
       'ERROR - movement_process-ERROR - ekg_quality_move_filter_0.289',
       'ERROR - movement_process-ERROR - ekg_quality_move_filter_0.219',
       'ERROR - movement_process-ERROR - ekg_quality_move_filter_0.278'],
      dtype=object)

In [14]:
df["message"].value_counts(dropna=False).head(30)

message
ERROR - properties_process-ERROR : There is no best segment                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

## 7. Message Column Transformation

In [15]:
def categorize_message(msg):
    if pd.isna(msg):
        return "no_message"
    if msg == "SUCCESS":
        return "success"
    if "movement_process" in msg:
        return "movement_failure"
    elif "ekg_quality_signal_filter" in msg:
        return "signal_quality_failure"
    elif "hr_process" in msg or "properties_process" in msg:
        return "hr_failure"
    elif "Traceback" in msg:
        return "system_error"
    else:
        return "other"

df["message_type"] = df["message"].apply(categorize_message)

In [16]:
df["message_type"].value_counts()

message_type
hr_failure                1154
signal_quality_failure     877
success                    735
system_error               272
no_message                  92
movement_failure             7
Name: count, dtype: int64

The majority of data segments fail at either signal quality or HR computation stages, with only a minority of segments (~23%) being fully usable. Additionally, a non-negligible portion of failures (~9%) are due to system-level errors rather than signal characteristics. Movement-related failures are minimal at the pipeline level.

## 8. Key Findings



### Data Structure
- The dataset is generated using sliding windows over ECG signals.
- Each row represents a time window where signal processing is applied.

### Pipeline Behavior
- The `state` column indicates execution success, not data validity.
- Processing consists of multiple stages: movement filtering → signal quality check → HR/HRV computation.

### Signal Quality
- Signal quality acts as a critical gate for downstream computations.
- A significant portion of data fails at the signal quality stage.
- Quality distribution suggests threshold-based behavior (low vs high quality).

### Movement
- Movement is mostly low (dominantly category 1).
- Movement rarely causes direct pipeline failure but may affect signal quality indirectly.

### HR and HRV Computation
- Even when signal quality passes, HR/HRV computation frequently fails.
- Errors such as “no best segment” indicate insufficient clean signal within windows.

### Message Analysis
- The `message` column encodes detailed pipeline failure reasons.
- Failures can be categorized into:
  - signal quality failures
  - HR computation failures
  - movement-related failures
  - system/runtime errors
- A notable portion of failures are due to system-level issues (tracebacks), not signal properties.

### Data Usability
- Only a minority of segments (~20–25%) result in fully successful processing.
- The majority of data is either unusable or partially processed.

### Key Insight for Product
- Signal quality and HR computation robustness are the primary bottlenecks.
- Any reporting or visualization should consider filtering for valid segments.
- Reporting should potentially include data quality indicators, not just physiological metrics.